<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Algorithmic Trading
## Chapter 13 · Online Algorithms and Streaming Decisions

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


### Notebook Goals

This notebook implements small, self-contained online algorithms that match the Chapter 13 discussion.

- Maintain streaming estimates of mean and variance with a single pass through the data.
- Implement a tiny epsilon-greedy bandit that chooses between two synthetic strategies.
- Visualise how estimates and action frequencies evolve over time.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")  # set notebook-wide plotting style
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("svg")
except Exception:
    pass
plt.rcParams.update({"figure.dpi": 300})


### 1. Online Mean and Variance

We start with a classic single-pass estimator using Welford-style updates. This pattern is handy whenever you cannot store the full return history.

In [ ]:
class OnlineMeanVar:
    """Streaming estimator for mean and variance of a series."""

    def __init__(self) -> None:
        self.n = 0  # number of observations seen
        self.mean = 0.0  # running mean
        self.M2 = 0.0  # sum of squared deviations from mean

    def update(self, x: float) -> None:
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        delta2 = x - self.mean
        self.M2 += delta * delta2

    @property
    def variance(self) -> float:
        if self.n < 2:
            return 0.0
        return self.M2 / (self.n - 1)


In [ ]:
rng = np.random.default_rng(seed=13)
stream = rng.normal(loc=0.0, scale=0.01, size=1_000)

est = OnlineMeanVar()
means = []
vars_ = []
for value in stream:
    est.update(float(value))
    means.append(est.mean)
    vars_.append(est.variance)

fig, (ax_m, ax_v) = plt.subplots(2, 1, figsize=(7.0, 4.6), sharex=True)
ax_m.plot(means)
ax_m.set_ylabel("mean estimate")
ax_m.grid(True, alpha=0.3)

ax_v.plot(vars_)
ax_v.set_ylabel("variance estimate")
ax_v.set_xlabel("step")
ax_v.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


### 2. Tiny Epsilon-Greedy Bandit

Next we sketch a two-armed bandit that might correspond to two simple strategies or feature sets. Rewards are simulated but the logic mirrors a live setup.

In [ ]:
def epsilon_greedy_bandit(
    mu: np.ndarray,
    sigma: np.ndarray,
    epsilon: float=0.1,
    steps: int=1_000,
) -> tuple[np.ndarray, np.ndarray]:
    """Simulate a tiny epsilon-greedy bandit with Gaussian rewards."""
    rng_local = np.random.default_rng(seed=13)
    n_arms = mu.shape[0]
    counts = np.zeros(n_arms, dtype=int)
    values = np.zeros(n_arms, dtype=float)

    actions = np.empty(steps, dtype=int)
    rewards = np.empty(steps, dtype=float)

    for t in range(steps):
        if rng_local.random() < epsilon or t < n_arms:
            arm = int(rng_local.integers(n_arms))
        else:
            arm = int(np.argmax(values))

        reward = float(rng_local.normal(mu[arm], sigma[arm]))
        counts[arm] += 1
        values[arm] += (reward - values[arm]) / counts[arm]

        actions[t] = arm
        rewards[t] = reward

    return actions, rewards


mu = np.array([0.0002, 0.0005])  # average rewards for arm 0 and 1
sigma = np.array([0.01, 0.012])  # reward volatility per arm
actions, rewards = epsilon_greedy_bandit(mu, sigma, epsilon=0.1, steps=2_000)

cum_rewards = rewards.cumsum()
freq_arm1 = (actions == 1).cumsum() / np.arange(1, actions.shape[0] + 1)

fig, (ax_r, ax_f) = plt.subplots(2, 1, figsize=(7.0, 4.6), sharex=True)
ax_r.plot(cum_rewards)
ax_r.set_ylabel("cumulative reward")
ax_r.grid(True, alpha=0.3)

ax_f.plot(freq_arm1)
ax_f.set_ylabel("share of pulls: arm 1")
ax_f.set_xlabel("step")
ax_f.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


### Takeaways

- Online estimators and bandits are natural companions to the streaming sandbox from Chapter 12.
- Many sophisticated methods start from patterns shown here: incremental updates and simple exploration–exploitation rules.
- When connecting to real tick data, replace the simulated rewards with strategy P&L but keep the update logic intact.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">